# HW04 · Нейросети и обучение / Neural networks and training


**Данные / Data:** [Digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html), реальные рукописные цифры из коллекции UCI / real handwritten digits from the UCI collection. 1,797 images, 8×8 pixels, intensities 0–16, 10 classes. Это не MNIST / This is not MNIST.

**Объект / Observation:** одно изображение / one image. **Признаки / Features:** 64 яркости, доступные при прогнозе / 64 intensities available at prediction time. **Цель / Target:** цифра 0–9 / digit 0–9.

**Ограничение / Limitation:** load_digits не содержит writer ID: разбиение по изображениям не доказывает перенос на новых авторов. / No writer IDs are supplied: an image split does not establish generalisation to unseen writers.

**Запуск / Runtime:** Google Colab, CPU, «Выполнить всё / Run all». Библиотеки Colab уже установлены / Colab includes the required libraries. Порядок: Train → Validation → зафиксированный выбор / frozen choice → Test.


**Сдача / Submission:** `Группа_Фамилия_HW04.ipynb` / `Group_Surname_HW04.ipynb` в [общую папку / submission folder](https://drive.google.com/drive/folders/1NNElGtsfynHtIHNngesVgpm71Hi9lkXm).

**Срок / Deadline:** [матрица групп / group matrix](https://docs.google.com/spreadsheets/d/1zzedKe77LfubpYsN4gsesqrvGwnb4pAoEuqLpkfNr6A/edit), HW04. Пусто = дата не задана / blank = not set.

**100 баллов / points:** код 80 + объяснения 20. Порог accuracy не задан / No accuracy threshold. Заполните TODO; затем `READY=True` и выполните все ячейки / Complete TODOs, set `READY=True`, then run all cells.

In [ ]:
import copy, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss, accuracy_score
SEED = 2026
CLASSES = np.arange(10)
print({'python': platform.python_version(), 'numpy': np.__version__,
       'sklearn': sklearn.__version__})
digits = load_digits()
X = digits.data.astype(np.float64) / 16.0  # Fixed format bound / Известная граница формата
y = digits.target
row_ids = np.arange(len(y))
assert X.shape == (1797, 64) and y.shape == (1797,)
assert 0 <= X.min() <= X.max() <= 1
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for label, ax in enumerate(axes.flat):
    i = np.flatnonzero(y == label)[0]
    ax.imshow(digits.images[i], cmap='gray_r', vmin=0, vmax=16)
    ax.set_title(str(label)); ax.axis('off')
plt.tight_layout(); plt.show()

## A · Данные и разбиение / Data and split · 15

Создайте train_ids, val_ids, test_ids как в P04: 20% Test, затем 25% от оставшегося на Validation, stratify, seed=2026. / Create the same stratified 60/20/20 split as P04. Покажите по одному примеру каждого класса; проверьте покрытие, отсутствие пересечений и все 10 классов в каждой части. / Display every class; verify coverage, disjointness and all ten labels per split.

In [ ]:
READY = False  # Set True after completing all TODOs / После выполнения всех TODO
train_ids = val_ids = test_ids = None
# TODO: split indices; define X_train, y_train, X_val, y_val

## B · Baseline · 10

Обучите LogisticRegression(C=1,max_iter=2000) только на Train; покажите Validation loss и accuracy. / Fit only on Train; report validation loss and accuracy.

In [ ]:
baseline = None
# TODO: fit baseline and evaluate on Validation

## C · Цикл обучения / Training loop · 25

Реализуйте train_run: 60 эпох partial_fit, ReLU, Adam, batch=64, alpha=0.0001, seed=2026, early_stopping=False. После каждой эпохи вычисляйте log_loss Train/Validation и accuracy Validation. / Implement 60 epochs and record train/validation cross-entropy and validation accuracy.

Возврат / Return: dict(model, history, width, lr, best_epoch, val_loss), history — DataFrame(epoch,train_loss,val_loss,val_accuracy). Сохраняйте deepcopy при строго меньшем Validation loss. / Retain a deep copy on strictly lower validation loss. Test не должен входить в функцию / Test must not enter the function.

In [ ]:
def train_run(X_train, y_train, X_val, y_val, width, lr, epochs=60, seed=SEED):
    # TODO: implement the loop and checkpoint selection / Цикл и выбор checkpoint
    raise NotImplementedError('Complete train_run / Реализуйте train_run')

## D · Сравнение / Comparison · 20

Запустите (16,.001), (16,.01), (64,.001), (64,.01). Заполните runs, best, summary, selection. / Run the four configurations and populate these variables.

Постройте обе кривые всех четырёх запусков, отметьте выбранные эпохи. Выберите best по min Validation loss; при равенстве — первый вариант. / Plot both curves for all runs, mark selected epochs, choose minimum validation loss (first config on ties).

selection: width, lr, best_epoch, val_loss, criterion. Сохраните таблицу и решение до Test / Record your table and decision before Test.

In [ ]:
configs = [(16, 0.001), (16, 0.01), (64, 0.001), (64, 0.01)]
runs = []
best = summary = selection = None
# TODO: run, compare, plot and freeze the choice

## E · Финальная оценка / Final evaluation · 10

После фиксации selection сравните baseline и best["model"] на Test: loss, accuracy, вероятности формы (n_test,10) с суммой строк 1. / After freezing selection, report test loss, accuracy and valid probabilities for both models. Создайте results как список словарей model,test_loss,test_accuracy / Create results as a list of dictionaries. Не меняйте настройки после Test / Do not retune after Test.

In [ ]:
results = []
# TODO: final evaluation only after frozen selection

## F · Объяснения / Explanations · 20

По 5 баллов / 5 points each:
1. Сравните кривые ваших четырёх запусков, объясните выбор эпохи. / Interpret your four runs and the selected epoch.
2. Объясните различие loss/accuracy на ваших результатах. / Explain loss versus accuracy using your results.
3. Что показывает сравнение learning rate при одинаковой ширине? / What does changing learning rate at fixed width show?
4. Почему нельзя обещать перенос на новых авторов и подбирать настройки по Test? / Why can you neither claim unseen-writer generalisation nor tune on Test?

Для проекта: источник данных, единица разбиения, доступность признаков. / For your project: data source, split unit, feature availability.

**Ответы / Answers:**

1. TODO
2. TODO
3. TODO
4. TODO

Проект / Project: TODO

## Самопроверка / Self-check

Проверки не оценивают объяснения и не доказывают отсутствие утечки; преподаватель проверяет протокол отдельно. / Checks do not assess explanations or establish absence of leakage; the protocol is reviewed separately.

In [ ]:
def check_work():
    assert len(train_ids) == 1077 and len(val_ids) == 360 and len(test_ids) == 360
    parts = [set(train_ids), set(val_ids), set(test_ids)]
    assert all(not parts[i] & parts[j] for i in range(3) for j in range(i + 1, 3))
    assert set.union(*parts) == set(row_ids)
    assert all(set(y[list(p)]) == set(CLASSES) for p in parts)
    assert baseline is not None and len(runs) == 4
    assert [(r['width'], r['lr']) for r in runs] == configs
    for r in runs:
        h = r['history']
        assert list(h.epoch) == list(range(1, 61))
        assert np.isfinite(h[['train_loss', 'val_loss', 'val_accuracy']].to_numpy()).all()
        assert r['best_epoch'] == int(h.loc[h.val_loss.idxmin(), 'epoch'])
        assert np.isclose(r['val_loss'], h.val_loss.min())
        # Verify the retained checkpoint itself / Проверяем сам сохранённый checkpoint
        assert np.isclose(log_loss(y_val, r['model'].predict_proba(X_val), labels=CLASSES), r['val_loss'])
        assert sum(a.size for a in r['model'].coefs_ + r['model'].intercepts_) == 75*r['width'] + 10
    assert best is min(runs, key=lambda r: r['val_loss'])
    assert all(selection[k] == best[k] for k in ['width', 'lr', 'best_epoch', 'val_loss'])
    assert len(results) == 2
    assert all(0 <= r['test_accuracy'] <= 1 and np.isfinite(r['test_loss']) for r in results)
    print('Structural checks passed / Структурные проверки пройдены. This is not a grade / Это не оценка.')
if READY:
    check_work()
else:
    print('Complete TODOs, set READY=True and run all / Заполните TODO, READY=True, выполните всё')